# RISS 종합목록 API 데이터 수집 및 분석 실습

**목적:** 기관별 국내·국외 MARC 서지 구축 현황 수집 → pandas 분석 → Excel 저장.

이 API는 논문명·저자 검색 API가 아닙니다. 개인 OpenAI API 키도 필요하지 않습니다.

**주의:** `RISS_API_URL`은 승인된 공공데이터포털 Swagger의 `GET` 경로를 등록해야 합니다. 테스트용 가짜 데이터를 실제 검색결과로 오해하지 마세요.

In [ ]:
# 셀 1. 라이브러리 준비
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openpyxl import load_workbook

from riss_catalog import (
    normalize_riss_rows, fetch_riss_catalog,
    summarize_riss, make_riss_excel, validate_endpoint,
)

load_dotenv()
API_KEY = os.getenv("RISS_API_KEY") or os.getenv("DATA_GO_KR_API_KEY", "")
RISS_API_URL = os.getenv("RISS_API_URL", "")
print("실습 모듈 준비 완료")
print("API 키 설정:", "있음" if API_KEY else "없음")
print("RISS GET URL 설정:", "있음" if RISS_API_URL else "없음")

## 1단계 — 샘플 데이터로 JSON 정리 원리 이해하기

In [ ]:
# 셀 2. 공식 데이터 설명에 맞춘 가상 데이터 (실제 RISS 자료가 아님)
sample_records = [
    {"기관코드": "TEST-A", "조사년도": "2022", "데이터 국내 구축건수": "1,200", "데이터 국외 구축건수": "200"},
    {"기관코드": "TEST-B", "조사년도": "2022", "데이터 국내 구축건수": "800", "데이터 국외 구축건수": "100"},
]
sample_df, sample_columns = normalize_riss_rows(sample_records)
display(sample_df)

In [ ]:
# 셀 3. 별도 assert 검증 셀 (샘플용)
assert len(sample_df) == 2, "샘플 데이터가 2건이어야 합니다."
assert sample_df.loc[0, "합계 구축건수"] == 1400
assert sample_df["국내 구축건수"].sum() == 2000
print("가상 데이터 구조 검증 통과")

## 2단계 — 승인된 실제 RISS 종합목록 API 조회하기

In [ ]:
# 셀 4. 실제 API 요청 (인증키와 GET URL이 있어야 실행)
real_df = pd.DataFrame()
real_total = 0

if not API_KEY:
    print(".env 파일에 DATA_GO_KR_API_KEY 또는 RISS_API_KEY를 입력하세요.")
elif not RISS_API_URL:
    print("RISS_API_URL이 없습니다. 승인된 Swagger의 GET 전체 호출주소를 등록하세요.")
else:
    validate_endpoint(RISS_API_URL)
    real_total, real_df, real_columns = fetch_riss_catalog(
        API_KEY, RISS_API_URL, pages=2, per_page=50,
    )
    print("API 전체 기록:", real_total)
    print("실제 수집한 기록:", len(real_df))
    print("확인된 필드명:", real_columns)
    display(real_df.head())

In [ ]:
# 셀 5. 실제 API 검증 셀
# API URL/인증키가 아직 없는 경우 실습 준비 상태를 표시합니다.
if API_KEY and RISS_API_URL:
    assert real_total >= len(real_df)
    assert isinstance(real_df, pd.DataFrame)
    assert {"기관코드", "조사년도", "국내 구축건수", "국외 구축건수"}.issubset(real_df.columns)
    print("실제 API 응답 검증 통과")
else:
    print("실제 API 검증 보류: 인증키 및 RISS_API_URL 설정 후 실행하세요.")

## 3단계 — pandas 집계와 Excel 과제 파일 만들기

In [ ]:
# 셀 6. 실제 수집 데이터만 분석합니다.
if real_df.empty:
    print("실제 수집 자료가 없어 분석/엑셀 저장을 건너뜁니다.")
    saved_path = None
else:
    by_year, by_org = summarize_riss(real_df)
    print("조사년도별 통계")
    display(by_year)
    print("기관별 통계 (상위 10건)")
    display(by_org.head(10))
    saved_path = Path("RISS_기관별서지구축_분석.xlsx")
    saved_path.write_bytes(make_riss_excel(real_df, by_year, by_org))
    print("Excel 저장 완료:", saved_path.resolve())

In [ ]:
# 셀 7. Excel 저장 및 내용 검증
if saved_path is not None:
    assert saved_path.exists(), "Excel 파일이 생성되지 않았습니다."
    workbook = load_workbook(saved_path, read_only=True)
    assert workbook.sheetnames == ["수집원본", "연도별통계", "기관별통계", "출처및유의사항"]
    assert workbook["수집원본"].max_row == len(real_df) + 1
    assert by_year["기관기록 수"].sum() == len(real_df)
    print("최종 Excel 검증 통과")
else:
    print("검증 보류: 실제 API 데이터 수집 후 검증할 수 있습니다.")

### 참고 자료
- 데이터 출처: https://www.data.go.kr/data/15071949/fileData.do
- Swagger URL에 있는 `GET` 경로를 모르면 **인증키를 공유하지 말고 GET 줄만 확인**하세요.
- API 원본 건수와 실제 수집 건수는 다를 수 있습니다. 이 자료의 구축건수는 학술논문 발행건수가 아닙니다.